In [23]:
import pypsa
import pandas as pd
import numpy as np

In [34]:
raw_2 = pd.read_csv("household_data_60min_singleindex.csv")
print("a")

a


In [ ]:
raw_dataset = pd.read_csv("household_data_60min_singleindex.csv",
                          parse_dates=["utc_timestamp"],
                          index_col="utc_timestamp")


start_time = "2016-07-15 00:00:00"
end_time = "2016-07-15 23:00:00"
dataset_day = raw_dataset.loc[start_time:end_time]
print(dataset_day)

load_cols = [col for col in dataset_day.columns if "grid_import" in col and "residential" in col]
pv_cols = [col for col in dataset_day.columns if "pv" in col and "residential" in col]

if (len(load_cols) == 0 or len(pv_cols) == 0):
    raise ValueError("Could not find consumption or solar columns")

print(f"Found {len(load_cols)} loads and {len(pv_cols)} pv")

                                 cet_cest_timestamp  \
utc_timestamp                                         
2016-07-15 00:00:00+00:00  2016-07-15T02:00:00+0200   
2016-07-15 01:00:00+00:00  2016-07-15T03:00:00+0200   
2016-07-15 02:00:00+00:00  2016-07-15T04:00:00+0200   
2016-07-15 03:00:00+00:00  2016-07-15T05:00:00+0200   
2016-07-15 04:00:00+00:00  2016-07-15T06:00:00+0200   
2016-07-15 05:00:00+00:00  2016-07-15T07:00:00+0200   
2016-07-15 06:00:00+00:00  2016-07-15T08:00:00+0200   
2016-07-15 07:00:00+00:00  2016-07-15T09:00:00+0200   
2016-07-15 08:00:00+00:00  2016-07-15T10:00:00+0200   
2016-07-15 09:00:00+00:00  2016-07-15T11:00:00+0200   
2016-07-15 10:00:00+00:00  2016-07-15T12:00:00+0200   
2016-07-15 11:00:00+00:00  2016-07-15T13:00:00+0200   
2016-07-15 12:00:00+00:00  2016-07-15T14:00:00+0200   
2016-07-15 13:00:00+00:00  2016-07-15T15:00:00+0200   
2016-07-15 14:00:00+00:00  2016-07-15T16:00:00+0200   
2016-07-15 15:00:00+00:00  2016-07-15T17:00:00+0200   
2016-07-15

Create a PyPSA network and set the snapshots for a period of 24 h.

In [25]:
network = pypsa.Network()
timestamps = pd.date_range(start_time, periods=24, freq="h")
network.set_snapshots(timestamps)


Add the buses for the IEEE 33-bus system. The base voltage is 12.66 kV

In [26]:
for i in range(1, 34):
    network.add(
        "Bus",
        f"Bus_{i}",
        v_nom=12.66
    )

Adding the lines. Resistance (r) and reactance (x) are in Ohm. The thermal capacity (s_nom) is set high so as to not be a limiting factor

In [27]:
# (from, to, r, x)
line_data = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (from_b, to_b, r, x) in enumerate(line_data):
    network.add(
        "Line", 
        f"Line_{from_b}-{to_b}",
        bus0=f"Bus_{from_b}",
        bus1=f"Bus_{to_b}",
        r=r,
        x=x,
        s_nom=5000
    )

Add the loads. p_set is the active power (kW) and q_set is the reactive power (kVAr)

In [28]:
load_data = [
    (100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]

Define all the profiles

In [29]:
# raw_load_profile = np.sin(np.linspace(0, 2 * np.pi, 24)) * 0.4 + 0.6
price_profile = np.array([50, 45, 40, 40, 42, 48, 60, 75, 70, 65, 50, 30,
                          20, 20, 25, 45, 65, 80, 90, 100, 85, 70, 60, 55])
grid_price_series = pd.Series(price_profile, index=network.snapshots)

# raw_pv_profile = np.sin(np.linspace(-np.pi / 2, 3 * np.pi / 2, 24))
# raw_pv_profile[raw_pv_profile < 0] = 0
# pv_series = pd.Series(raw_pv_profile, index=network.snapshots)


Apply the time varying profiles

In [31]:
for i, (p, q) in enumerate(load_data):

    raw_load_col = load_cols[i % len(load_cols)]
    raw_load_series = dataset_day[raw_load_col]
    normalized_load = raw_load_series / raw_load_series.max()
    real_load_profile = normalized_load * p

    # load_profile_series = pd.Series(p * raw_load_profile, index=network.snapshots)
    network.add(
        "Load",
        f"Load_bus_{i+2}",
        bus=f"Bus_{i+2}",
        p_set=real_load_profile,
        q_set=q
    )

ValueError: Series p_set has an index which does not align with the passed network snapshots.

Add grid, PV and batteries

In [ ]:
network.add(
    "Generator",
    "Substation",
    bus="Bus_1",
    p_nom=10000,
    p_min_pu=-1.0,
    carrier="gas",
    marginal_cost=grid_price_series
)

for i in range(2, 34):

    raw_pv_col = pv_cols[i % len(pv_cols)]
    raw_pv_series = dataset_day[raw_pv_col]
    real_pv_profile = raw_pv_series / raw_pv_series.max()
    real_pv_profile = real_pv_profile.fillna(0)

    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=150,
        p_max_pu=real_pv_profile,
        carrier="solar",
        marginal_cost=0
    )
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=200, # Nominal capacity in kW
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01 # 1% loss per hour
    )

In [ ]:
network.optimize()

Index(['Bus_1', 'Bus_2', 'Bus_3', 'Bus_4', 'Bus_5', 'Bus_6', 'Bus_7', 'Bus_8',
       'Bus_9', 'Bus_10', 'Bus_11', 'Bus_12', 'Bus_13', 'Bus_14', 'Bus_15',
       'Bus_16', 'Bus_17', 'Bus_18', 'Bus_19', 'Bus_20', 'Bus_21', 'Bus_22',
       'Bus_23', 'Bus_24', 'Bus_25', 'Bus_26', 'Bus_27', 'Bus_28', 'Bus_29',
       'Bus_30', 'Bus_31', 'Bus_32', 'Bus_33'],
      dtype='str', name='Bus')
Index(['Line_1-2', 'Line_2-3', 'Line_3-4', 'Line_4-5', 'Line_5-6', 'Line_6-7',
       'Line_7-8', 'Line_8-9', 'Line_9-10', 'Line_10-11', 'Line_11-12',
       'Line_12-13', 'Line_13-14', 'Line_14-15', 'Line_15-16', 'Line_16-17',
       'Line_17-18', 'Line_2-19', 'Line_19-20', 'Line_20-21', 'Line_21-22',
       'Line_3-23', 'Line_23-24', 'Line_24-25', 'Line_6-26', 'Line_26-27',
       'Line_27-28', 'Line_28-29', 'Line_29-30', 'Line_30-31', 'Line_31-32',
       'Line_32-33'],
      dtype='str', name='Line')
Index(['Substation', 'PV_bus_2', 'PV_bus_3', 'PV_bus_4', 'PV_bus_5',
       'PV_bus_6', 'PV_bus_7', 'P

('ok', 'optimal')

In [ ]:
print(f"cost: {network.objective:.2f}")


cost: -7781.96


In [ ]:
network.generators_t.p

Generator,Substation,PV_bus_2,PV_bus_3,PV_bus_4,PV_bus_5,PV_bus_6,PV_bus_7,PV_bus_8,PV_bus_9,PV_bus_10,...,PV_bus_24,PV_bus_25,PV_bus_26,PV_bus_27,PV_bus_28,PV_bus_29,PV_bus_30,PV_bus_31,PV_bus_32,PV_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 01:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 02:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 03:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 04:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 05:00:00,5000.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-01-01 06:00:00,3383.972207,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,...,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362,10.236362
2025-01-01 07:00:00,-4378.222408,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,...,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942,50.231942
2025-01-01 08:00:00,-1835.635580,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,...,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048,86.502048


In [ ]:
network.storage_units_t.p

StorageUnit,Battery_bus_2,Battery_bus_3,Battery_bus_4,Battery_bus_5,Battery_bus_6,Battery_bus_7,Battery_bus_8,Battery_bus_9,Battery_bus_10,Battery_bus_11,...,Battery_bus_24,Battery_bus_25,Battery_bus_26,Battery_bus_27,Battery_bus_28,Battery_bus_29,Battery_bus_30,Battery_bus_31,Battery_bus_32,Battery_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,-133.778636,-200.000000,-200.000000,-200.000000,0.000000,0.000000,0.000000,0.000000,-200.000000,-134.702503,...,-200.000000,-200.000000,0.000000,0.000000,0.000000,0.000000,-200.000000,-200.000000,0.000000,0.000000
2025-01-01 01:00:00,-190.669558,-200.000000,-200.000000,-200.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-200.000000,-127.151016,0.000000,0.000000,0.000000,0.000000,-127.151016,0.000000,0.000000,0.000000
2025-01-01 02:00:00,0.000000,-127.879506,0.000000,0.000000,-200.000000,-200.000000,-53.516417,0.000000,-200.000000,0.000000,...,0.000000,-200.000000,-200.000000,0.000000,0.000000,0.000000,-200.000000,0.000000,0.000000,0.000000
2025-01-01 03:00:00,-200.000000,0.000000,0.000000,0.000000,0.000000,-59.042063,0.000000,-52.981253,0.000000,-136.866190,...,0.000000,0.000000,-59.042063,0.000000,-55.021861,0.000000,0.000000,-62.982263,0.000000,-200.000000
2025-01-01 04:00:00,0.000000,0.000000,-125.334704,-125.334704,-58.451643,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-200.000000,-200.000000,-200.000000,0.000000,0.000000,-52.451440,0.000000
2025-01-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-200.000000,-200.000000,-122.140759,0.000000,...,0.000000,0.000000,0.000000,-53.926926,0.000000,-53.926926,0.000000,0.000000,-200.000000,-55.906926
2025-01-01 06:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-01-01 07:00:00,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,...,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
2025-01-01 08:00:00,200.000000,200.000000,200.000000,200.000000,0.000000,0.000000,0.000000,0.000000,200.000000,8.108068,...,102.479227,200.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,0.000000,0.000000
